# MIDI highest-note extraction

This notebook reads the MIDI files in `train`, extracts note-on events with `mido`, aligns each recording to its first note, groups nearly simultaneous onsets into time steps, and prints the highest note from each step for every full song.

The output note string uses one unique printable ASCII character per MIDI note and no separators. The later compression experiments operate only on those generated strings, not on the MIDI files or note lists.


In [1]:
from collections import Counter
from itertools import combinations
from pathlib import Path
import base64
import zlib

import mido

TRAIN_DIR = Path("train")
ONSET_GROUP_SECONDS = 0.08
PRINTABLE_ASCII = [chr(code) for code in range(33, 127)]

NOTE_NAMES = ("C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B")


def note_name(note):
    """Convert a MIDI note number to a scientific pitch name.

    Args:
        note: MIDI note number, where middle C is 60.

    Returns:
        A pitch-name string such as "C4" or "F#5".
    """
    return f"{NOTE_NAMES[note % 12]}{note // 12 - 1}"


def edit_distance(left, right):
    """Compute the Levenshtein edit distance between two sequences.

    Args:
        left: First sequence to compare.
        right: Second sequence to compare.

    Returns:
        The minimum number of insertions, deletions, or substitutions needed to transform `left` into `right`.
    """
    previous = list(range(len(right) + 1))
    for i, left_item in enumerate(left, start=1):
        current = [i]
        for j, right_item in enumerate(right, start=1):
            current.append(
                min(
                    previous[j] + 1,
                    current[j - 1] + 1,
                    previous[j - 1] + (left_item != right_item),
                )
            )
        previous = current
    return previous[-1]


def extract_note_onsets(path):
    """Extract note-on events from a MIDI file with absolute times in seconds.

    Args:
        path: Path to a MIDI file.

    Returns:
        A list of dictionaries, each containing the onset time in seconds, MIDI note number, note name, and velocity.
    """
    midi = mido.MidiFile(path)
    tempo = 500_000
    seconds = 0.0
    note_onsets = []

    for message in mido.merge_tracks(midi.tracks):
        seconds += mido.tick2second(message.time, midi.ticks_per_beat, tempo)

        if message.type == "set_tempo":
            tempo = message.tempo
        elif message.type == "note_on" and message.velocity > 0:
            note_onsets.append(
                {
                    "time": seconds,
                    "note": message.note,
                    "name": note_name(message.note),
                    "velocity": message.velocity,
                }
            )

    return note_onsets


def highest_notes_by_onset_group(note_onsets, onset_group_seconds=ONSET_GROUP_SECONDS):
    """Select the highest note from each onset group across an entire song.

    Args:
        note_onsets: Note-on dictionaries returned by `extract_note_onsets`.
        onset_group_seconds: Maximum time span from the first onset in a group for notes to be treated as the same time step.

    Returns:
        A list of MIDI note numbers, one highest note per onset group for the full song.
    """
    if not note_onsets:
        return []

    start_time = note_onsets[0]["time"]
    aligned_onsets = [{**note, "aligned_time": note["time"] - start_time} for note in note_onsets]

    groups = []
    current_group = []
    group_start = None

    for onset in aligned_onsets:
        if not current_group or onset["aligned_time"] - group_start <= onset_group_seconds:
            if not current_group:
                group_start = onset["aligned_time"]
            current_group.append(onset)
        else:
            groups.append(current_group)
            current_group = [onset]
            group_start = onset["aligned_time"]

    if current_group:
        groups.append(current_group)

    return [max(group, key=lambda onset: onset["note"])["note"] for group in groups]


def build_note_character_map(song_note_lists):
    """Build a stable one-character ASCII encoding for all notes in the songs.

    Args:
        song_note_lists: Mapping from song name to extracted MIDI note-number list.

    Returns:
        A dictionary mapping each MIDI note number to a unique printable ASCII character.
    """
    unique_notes = sorted({note for notes in song_note_lists.values() for note in notes})
    if len(unique_notes) > len(PRINTABLE_ASCII):
        raise ValueError(f"Need {len(unique_notes)} characters, but only {len(PRINTABLE_ASCII)} printable ASCII characters are available")
    return {note: PRINTABLE_ASCII[index] for index, note in enumerate(unique_notes)}


def encode_notes_as_ascii(notes, note_character_map):
    """Encode a note list as a separator-free ASCII string.

    Args:
        notes: Sequence of MIDI note numbers.
        note_character_map: Mapping from MIDI note number to unique ASCII character.

    Returns:
        A string containing one ASCII character per note and no separators.
    """
    return "".join(note_character_map[note] for note in notes)


def print_string_report(label, strings_by_song):
    """Print length and pairwise similarity statistics for generated strings.

    Args:
        label: Human-readable label for the strings being reported.
        strings_by_song: Mapping from song name to generated string.

    Returns:
        None. The function prints a compact report.
    """
    print(label)
    for song_name, value in strings_by_song.items():
        preview = value[:120]
        suffix = "..." if len(value) > len(preview) else ""
        print(f"  {song_name}: length={len(value)}, preview={preview!r}{suffix}")

    exact_match = len(set(strings_by_song.values())) <= 1
    print(f"  Exact match across songs: {exact_match}")
    for (left_name, left_value), (right_name, right_value) in combinations(strings_by_song.items(), 2):
        distance = edit_distance(left_value, right_value)
        denominator = max(len(left_value), len(right_value), 1)
        print(f"  edit distance {left_name} <-> {right_name}: {distance} ({distance / denominator:.3%} of longer string)")


def collapse_consecutive_repeats(note_string):
    """Collapse every run of identical adjacent characters to one character.

    Args:
        note_string: Separator-free ASCII note string.

    Returns:
        A shorter string where repeated adjacent notes are represented once.
    """
    if not note_string:
        return ""
    collapsed = [note_string[0]]
    for character in note_string[1:]:
        if character != collapsed[-1]:
            collapsed.append(character)
    return "".join(collapsed)


def run_length_encode(note_string):
    """Encode repeated adjacent characters as character-count runs.

    Args:
        note_string: Separator-free ASCII note string.

    Returns:
        A run-length encoded string where a single character is unchanged and a run is stored as character plus decimal count.
    """
    if not note_string:
        return ""

    encoded = []
    current_character = note_string[0]
    count = 1

    for character in note_string[1:]:
        if character == current_character:
            count += 1
        else:
            encoded.append(current_character if count == 1 else f"{current_character}{count}")
            current_character = character
            count = 1

    encoded.append(current_character if count == 1 else f"{current_character}{count}")
    return "".join(encoded)


def collapse_repeated_chunks(note_string, chunk_size):
    """Collapse adjacent repeated chunks of a fixed length to one copy.

    Args:
        note_string: Separator-free ASCII note string.
        chunk_size: Number of characters in each chunk to compare.

    Returns:
        A string where immediately repeated chunks of `chunk_size` are represented once.
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    output = []
    previous_chunk = None
    index = 0

    while index < len(note_string):
        chunk = note_string[index : index + chunk_size]
        if chunk != previous_chunk:
            output.append(chunk)
            previous_chunk = chunk
        index += chunk_size

    return "".join(output)


def collapse_repeated_motifs(note_string, min_chunk_size=2, max_chunk_size=8):
    """Collapse adjacent repeated motifs using several chunk sizes.

    Args:
        note_string: Separator-free ASCII note string.
        min_chunk_size: Smallest motif length to try.
        max_chunk_size: Largest motif length to try.

    Returns:
        The shortest string found after applying fixed-size repeated-chunk collapse across the requested motif sizes.
    """
    candidates = [collapse_repeated_chunks(note_string, chunk_size) for chunk_size in range(min_chunk_size, max_chunk_size + 1)]
    return min(candidates, key=len) if candidates else note_string


def contour_signature(note_string):
    """Convert a note string to a compressed up/down/same contour signature.

    Args:
        note_string: Separator-free ASCII note string whose character order follows ascending pitch.

    Returns:
        A string over `U`, `D`, and `S` with consecutive repeated contour symbols collapsed.
    """
    if len(note_string) < 2:
        return note_string

    contour = []
    for previous, current in zip(note_string, note_string[1:]):
        if current > previous:
            contour.append("U")
        elif current < previous:
            contour.append("D")
        else:
            contour.append("S")

    return collapse_consecutive_repeats("".join(contour))


def modal_block_signature(note_string, block_size=8):
    """Represent each fixed-size block by its most common character.

    Args:
        note_string: Separator-free ASCII note string.
        block_size: Number of characters per block.

    Returns:
        A string containing one modal character per block.
    """
    if block_size <= 0:
        raise ValueError("block_size must be positive")

    signature = []
    for index in range(0, len(note_string), block_size):
        block = note_string[index : index + block_size]
        most_common_character = Counter(block).most_common(1)[0][0]
        signature.append(most_common_character)
    return "".join(signature)


def zlib_base85_encode(note_string):
    """Compress a note string with zlib and encode the bytes as ASCII.

    Args:
        note_string: Separator-free ASCII note string.

    Returns:
        An ASCII string containing the base85 representation of the zlib-compressed bytes.
    """
    compressed = zlib.compress(note_string.encode("ascii"), level=9)
    return base64.b85encode(compressed).decode("ascii")

## Full-song highest-note strings

This cell processes the whole song for each MIDI file. It prints the extracted MIDI note list, the note names, and the separator-free ASCII note string. The ASCII mapping is built from all notes observed in these songs so each distinct MIDI note gets exactly one unique character.


In [2]:
midi_files = sorted(TRAIN_DIR.glob("*.mid*"))
print(f"Found {len(midi_files)} MIDI files:")
for path in midi_files:
    print(f"- {path}")

song_note_onsets = {path.name: extract_note_onsets(path) for path in midi_files}
song_note_lists = {song_name: highest_notes_by_onset_group(note_onsets) for song_name, note_onsets in song_note_onsets.items()}
note_character_map = build_note_character_map(song_note_lists)
character_note_map = {character: note for note, character in note_character_map.items()}
song_note_strings = {song_name: encode_notes_as_ascii(note_list, note_character_map) for song_name, note_list in song_note_lists.items()}

print(f"Unique extracted MIDI notes: {len(note_character_map)}")
print("Note -> ASCII character map:")
for note, character in note_character_map.items():
    print(f"  {note:>3} {note_name(note):>3} -> {character!r}")

for song_name, note_list in song_note_lists.items():
    note_names = [note_name(note) for note in note_list]
    note_string = song_note_strings[song_name]

    print("=" * 88)
    print(song_name)
    print(f"  MIDI notes ({len(note_list)}): {note_list}")
    print(f"  Note names  ({len(note_names)}): {note_names}")
    print(f"  ASCII note string ({len(note_string)}): {note_string!r}")

print("=" * 88)
print_string_report("Raw separator-free ASCII note strings", song_note_strings)

Found 3 MIDI files:
- train/MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi
- train/MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi
- train/MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi


Unique extracted MIDI notes: 71
Note -> ASCII character map:
   30 F#1 -> '!'
   31  G1 -> '"'
   32 G#1 -> '#'
   33  A1 -> '$'
   34 A#1 -> '%'
   35  B1 -> '&'
   36  C2 -> "'"
   37 C#2 -> '('
   38  D2 -> ')'
   39 D#2 -> '*'
   40  E2 -> '+'
   41  F2 -> ','
   42 F#2 -> '-'
   43  G2 -> '.'
   44 G#2 -> '/'
   45  A2 -> '0'
   46 A#2 -> '1'
   47  B2 -> '2'
   48  C3 -> '3'
   49 C#3 -> '4'
   50  D3 -> '5'
   51 D#3 -> '6'
   52  E3 -> '7'
   53  F3 -> '8'
   54 F#3 -> '9'
   55  G3 -> ':'
   56 G#3 -> ';'
   57  A3 -> '<'
   58 A#3 -> '='
   59  B3 -> '>'
   60  C4 -> '?'
   61 C#4 -> '@'
   62  D4 -> 'A'
   63 D#4 -> 'B'
   64  E4 -> 'C'
   65  F4 -> 'D'
   66 F#4 -> 'E'
   67  G4 -> 'F'
   68 G#4 -> 'G'
   69  A4 -> 'H'
   70 A#4 -> 'I'
   71  B4 -> 'J'
   72  C5 -> 'K'
   73 C#5 -> 'L'
   74  D5 -> 'M'
   75 D#5 -> 'N'
   76  E5 -> 'O'
   77  F5 -> 'P'
   78 F#5 -> 'Q'
   79  G5 -> 'R'
   80 G#5 -> 'S'
   81  A5 -> 'T'
   82 A#5 -> 'U'
   83  B5 -> 'V'
   84  C6 -> 'W'
   8

  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: 563 (7.102% of longer string)


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 6310 (80.086% of longer string)


  edit distance MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 6355 (80.169% of longer string)


## Compression experiment 1: collapse adjacent repeated notes

This lossy string-only transform collapses every run of the same note character to a single character. It targets repeated notes or tremolo-like passages where a later matcher may care more about melodic shape than repeated strikes.


In [3]:
experiment_1_strings = {song_name: collapse_consecutive_repeats(note_string) for song_name, note_string in song_note_strings.items()}
print_string_report("Experiment 1: collapse adjacent repeated notes", experiment_1_strings)

Experiment 1: collapse adjacent repeated notes
  MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi: length=6987, preview='?ACDFGIKMODPQRSRNMKRW753WVYZ653Z_/^\\ZXURPLIFD@=;8641/./1343:;=?@?:F;G=I?KAMBNFRISIUKWMYNZS_UaWc`ZBF\\ZYWUSRPNMKJGFDBA?>?:'...
  MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: length=7052, preview='?ACDFGIKMODPQRSRNMKRW653WVYZ653Z_/^\\ZXWUSRPNLIFB?<:86431/./1343:;=?@?:F;G=I?KAMBNFRPSIUKWMYNZS_UaWc^=F\\ZYWUSRPNMKJGFDBA?'...
  MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: length=3094, preview='?ACDFGIKMODPQRSRNMKRMRW1W653WVYZ653Z_/^\\ZXWUSPNKGFCA>:8641/./1343:;=?@?:F;G=I?KAMBNFRGSIUKWMYNZS_UaWc^F\\ZYWUSRPNMKGFDBA?'...
  Exact match across songs: False


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: 577 (8.182% of longer string)


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 5532 (79.176% of longer string)


  edit distance MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 5598 (79.382% of longer string)


## Compression experiment 2: run-length encode repeated notes

This reversible-looking string-only transform keeps repeated-note information by replacing runs with the note character followed by the run count. It helps when long repeated runs exist, but can grow the string when most notes are not repeated.


In [4]:
experiment_2_strings = {song_name: run_length_encode(note_string) for song_name, note_string in song_note_strings.items()}
print_string_report("Experiment 2: run-length encode repeated notes", experiment_2_strings)

Experiment 2: run-length encode repeated notes
  MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi: length=7564, preview='?2A2CDFGIKMODP2Q3R3SRNMKR3W2753WV2YZ2653Z_/^\\ZXURPLIFD@=;8641/./1343:;=?@?:F;G=I?KAMBNFRISIUKWMYNZS_UaWc`ZBF\\ZYWUSRPNMKJ'...
  MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: length=7622, preview='?2A2CDFGIKMODP2Q3R3SRNMKR3W2653WV2YZ2653Z_/^\\ZXWUSRPNLIFB?<:86431/./1343:;=?@?:F;G=I?KAMBNFRPSIUKWMYNZS_UaWc^=F\\ZYWUSRPN'...
  MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: length=3221, preview='?2A2CDFGIKMODP2Q3R3SRNMKRMR2W1W653WV2YZ2653Z_/^\\ZXWUSPNKGFCA>:8641/./1343:;=?@?:F;G=I?KAMBNFRGSIUKWMYNZS_UaWc^F\\ZYWUSRPN'...
  Exact match across songs: False


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: 652 (8.554% of longer string)


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 6017 (79.548% of longer string)


  edit distance MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 6082 (79.795% of longer string)


## Compression experiment 3: collapse repeated two-note chunks

This string-only transform splits each song string into two-character chunks and collapses adjacent duplicate chunks to one copy. It targets repeated intervals or accompaniment fragments while staying more specific than single-character run collapse.


In [5]:
experiment_3_strings = {song_name: collapse_repeated_chunks(note_string, chunk_size=2) for song_name, note_string in song_note_strings.items()}
print_string_report("Experiment 3: collapse repeated two-note chunks", experiment_3_strings)

Experiment 3: collapse repeated two-note chunks
  MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi: length=7655, preview='??AACDFGIKMODPPQQQRRRSRNMKRRRWW753WVVYZZ653Z_/^\\ZXURPLIFD@=;8641/./1343:;=?@?:F;G=I?KAMBNFRISIUKWMYNZS_UaWc`ZBF\\ZYWUSRPN'...
  MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: length=7703, preview='??AACDFGIKMODPPQQQRRRSRNMKRRRWW653WVVYZZ653Z_/^\\ZXWUSRPNLIFB?<:86431/./1343:;=?@?:F;G=I?KAMBNFRPSIUKWMYNZS_UaWc^=F\\ZYWUS'...
  MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: length=3202, preview='??AACDFGIKMODPPQQQRRRSRNMKRMRRW1W653WVVYZZ653Z_/^\\ZXWUSPNKGFCA>:8641/./1343:;=?@?:F;G=I?KAMBNFRGSIUKWMYNZS_UaWc^F\\ZYWUSR'...
  Exact match across songs: False


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: 640 (8.308% of longer string)


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 6128 (80.052% of longer string)


  edit distance MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 6161 (79.982% of longer string)


## Compression experiment 4: collapse repeated motifs of length 2 through 8

This string-only transform tries repeated-chunk collapse for motif sizes from 2 to 8 characters and keeps the shortest result for each song. It looks for repeated short musical patterns without returning to MIDI data or numeric note lists.


In [6]:
experiment_4_strings = {song_name: collapse_repeated_motifs(note_string, min_chunk_size=2, max_chunk_size=8) for song_name, note_string in song_note_strings.items()}
print_string_report("Experiment 4: collapse repeated motifs of length 2 through 8", experiment_4_strings)

Experiment 4: collapse repeated motifs of length 2 through 8
  MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi: length=7655, preview='??AACDFGIKMODPPQQQRRRSRNMKRRRWW753WVVYZZ653Z_/^\\ZXURPLIFD@=;8641/./1343:;=?@?:F;G=I?KAMBNFRISIUKWMYNZS_UaWc`ZBF\\ZYWUSRPN'...
  MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: length=7703, preview='??AACDFGIKMODPPQQQRRRSRNMKRRRWW653WVVYZZ653Z_/^\\ZXWUSRPNLIFB?<:86431/./1343:;=?@?:F;G=I?KAMBNFRPSIUKWMYNZS_UaWc^=F\\ZYWUS'...
  MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: length=3194, preview='??AACDFGIKMODPPQQQRRRSRNMKRMRRW1W653WVVYZZ653Z_/^\\ZXWUSPNKGFCA>:8641/./1343:;=?@?:F;G=I?KAMBNFRGSIUKWMYNZS_UaWc^F\\ZYWUSR'...
  Exact match across songs: False


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: 640 (8.308% of longer string)


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 6142 (80.235% of longer string)


  edit distance MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 6173 (80.138% of longer string)


## Compression experiment 5: contour signature

This lossy string-only transform converts the note-character string into an up/down/same contour string, then collapses repeated contour directions. It is designed to make similar melodic shapes compare more closely even when exact notes or repeated articulations differ.


In [7]:
experiment_5_strings = {song_name: contour_signature(note_string) for song_name, note_string in song_note_strings.items()}
print_string_report("Experiment 5: contour signature", experiment_5_strings)

Experiment 5: contour signature
  MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi: length=5027, preview='SUSUDUSUSUSUDUSUSDUDSUSDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUD'...
  MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: length=5019, preview='SUSUDUSUSUSUDUSUSDUDSUSDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUD'...
  MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: length=2206, preview='SUSUDUSUSUSUDUDUSUDUDUDSUSDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDSDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDUDSUDUDUDUDUDUDUDUDUD'...
  Exact match across songs: False


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: 209 (4.158% of longer string)


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 2825 (56.197% of longer string)


  edit distance MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 2816 (56.107% of longer string)


## Compression experiment 6: modal fixed-size blocks

This lossy string-only transform divides each note string into eight-character blocks and represents each block by its most common character. It greatly reduces length while preserving a rough local pitch center for approximate matching.


In [8]:
experiment_6_strings = {song_name: modal_block_signature(note_string, block_size=8) for song_name, note_string in song_note_strings.items()}
print_string_report("Experiment 6: modal fixed-size blocks", experiment_6_strings)

Experiment 6: modal fixed-size blocks
  MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi: length=985, preview='?PRRV6ZD/;GIWaZM:?;:F??F?6668G=BBBDRB==KIABFIINLF===68===666(/3;8;SIDBNEIILLQILNS_>ZPUIZP1=6VSIIGG0F>K16WQK8\\YR???O=;U7O'...
  MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: length=991, preview='?PRRV6ZL13?KSZ=R?:;;=??F??6658==BBBDBB=NKDIAFINLLF==668;==666/G/;::8IIINEIILLQILNS_>ZPUNZP1A6VSIIGG00?L66WQK8\\YR???O=;U7'...
  MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: length=409, preview='?PRRWZ^N13?KSZFK??;=:??F?6668G=BBBDRB==KIABFBINLLF===68;==666///;8;5IBIEGIIELIJLENSBZPUNZP616ZSINGFG0?CQRWZ_S=3?=FMWV?:?'...
  Exact match across songs: False
  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: 483 (48.739% of longer string)
  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_

## Compression experiment 7: zlib plus base85

This reversible string-only transform compresses the ASCII note string with zlib and converts the compressed bytes back to ASCII with base85. It optimizes storage size, although it is less useful for approximate matching than the lossy musical signatures above.


In [9]:
experiment_7_strings = {song_name: zlib_base85_encode(note_string) for song_name, note_string in song_note_strings.items()}
print_string_report("Experiment 7: zlib plus base85", experiment_7_strings)

Experiment 7: zlib plus base85
  MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi: length=5024, preview='c-qxjiJGIhc7B#S(_3v|z<><KECvY>2m!{}l*`r2P4fP4kYCtUUEQ~5GPyTj@<sTUjt*KtIywsu!#Liiloh;gQYqWE?Yge-M9oo-qib8^et(=M$F`<t{d}4Z'...
  MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: length=5195, preview='c-qBT3uB`;w*D>KcG^xR!NwSYg?SnzKp;REV-v@j%yjSm|9@~l;UtrEy4~)5${gwF=tu%|o;VESc$YF>h`LGJw(B~jRIg+$(2gUqt?_U;O%ssmadA0M$8qRY'...
  MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: length=2037, preview='c-qBQjdtUz5q_5T+<SMEP3*{6SVoo*2!SL7Mi5zvHk&lvzW+0Huy=d1KRx$sPkSo)69Wu07Bk-nN*Sw#RGl_Q;oLZm-uvLK(cXAJCo@jbd^$ZY3kb{0`uS;j'...
  Exact match across songs: False


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi: 4917 (94.649% of longer string)


  edit distance MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 4513 (89.829% of longer string)


  edit distance MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.midi <-> MIDI-Unprocessed_XP_06_R2_2004_01_ORIG_MID--AUDIO_06_R2_2004_01_Track01_wav.midi: 4682 (90.125% of longer string)


## Compression experiment summary

This summary compares each experiment's total output length against the uncompressed separator-free ASCII note strings. Lower total length means less data for a later matching program to compare; the edit-distance columns show whether the compressed forms still agree closely across recordings.


In [13]:
compression_experiments = {
    "Raw separator-free ASCII": song_note_strings,
    "Experiment 1: collapse adjacent repeated notes": experiment_1_strings,
    "Experiment 2: run-length encode repeated notes": experiment_2_strings,
    "Experiment 3: collapse repeated two-note chunks": experiment_3_strings,
    "Experiment 4: collapse repeated motifs of length 2 through 8": experiment_4_strings,
    "Experiment 5: contour signature": experiment_5_strings,
    "Experiment 6: modal fixed-size blocks": experiment_6_strings,
    "Experiment 7: zlib plus base85": experiment_7_strings,
}

raw_total_length = sum(len(value) for value in song_note_strings.values())
summary_rows = []

for experiment_name, strings_by_song in compression_experiments.items():
    total_length = sum(len(value) for value in strings_by_song.values())
    saved_characters = raw_total_length - total_length
    compression_ratio = total_length / raw_total_length if raw_total_length else 0
    exact_match = len(set(strings_by_song.values())) <= 1
    pairwise_distances = [edit_distance(left_value, right_value) for (_, left_value), (_, right_value) in combinations(strings_by_song.items(), 2)]
    summary_rows.append(
        {
            "experiment": experiment_name,
            "total_length": total_length,
            "saved_characters": saved_characters,
            "compression_ratio": compression_ratio,
            "percent_smaller": 1 - compression_ratio,
            "exact_match": exact_match,
            "max_pairwise_edit_distance": max(pairwise_distances, default=0),
        }
    )

print(f"Raw total length: {raw_total_length} characters")
print("| Experiment | Total length | Saved vs raw | Percent smaller | Exact match | Max pairwise edit distance |")
print("|---|---:|---:|---:|---:|---:|")
for row in sorted(summary_rows, key=lambda item: item["total_length"]):
    print(f"| {row['experiment']} | {row['total_length']} | {row['saved_characters']} | {row['percent_smaller']:.1%} | {row['exact_match']} | {row['max_pairwise_edit_distance']} |")

best_row = min(summary_rows, key=lambda item: item["total_length"])
print(f"Shortest output: {best_row['experiment']} with {best_row['total_length']} total characters ({best_row['percent_smaller']:.1%} smaller than raw).")


Raw total length: 19076 characters
| Experiment | Total length | Saved vs raw | Percent smaller | Exact match | Max pairwise edit distance |
|---|---:|---:|---:|---:|---:|
| Experiment 6: modal fixed-size blocks | 2385 | 16691 | 87.5% | False | 830 |
| Experiment 5: contour signature | 12252 | 6824 | 35.8% | False | 2825 |
| Experiment 7: zlib plus base85 | 12256 | 6820 | 35.8% | False | 4917 |
| Experiment 1: collapse adjacent repeated notes | 17133 | 1943 | 10.2% | False | 5598 |
| Experiment 2: run-length encode repeated notes | 18407 | 669 | 3.5% | False | 6082 |
| Experiment 4: collapse repeated motifs of length 2 through 8 | 18552 | 524 | 2.7% | False | 6173 |
| Experiment 3: collapse repeated two-note chunks | 18560 | 516 | 2.7% | False | 6161 |
| Raw separator-free ASCII | 19076 | 0 | 0.0% | False | 6355 |
Shortest output: Experiment 6: modal fixed-size blocks with 2385 total characters (87.5% smaller than raw).
